# 0. Objective

Fetch all TRACE records for a given date from the OpenYLD API and
save them as a raw partitioned Parquet file.

Output path:
```
datastore/openyld/TRACE/raw/date=YYYY-MM-DD/data.parquet
```

Run daily (after market close / next morning) to build up a growing historical store.

## 0.1 Parameters

Set `TARGET_DATE` to the date you want to ingest.
Default is the previous business day (T-1).

In [1]:
import os
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # searches current dir and parents automatically

# Override this to ingest a specific date, e.g. TARGET_DATE = "2025-12-01"
TARGET_DATE = None

if TARGET_DATE is None:
    today = pd.Timestamp.today().normalize()
    TARGET_DATE = (today - pd.offsets.BDay(1)).strftime("%Y-%m-%d")

DATASTORE = Path("../datastore/openyld/TRACE/raw")

print(f"Target date : {TARGET_DATE}")
print(f"Output path : {DATASTORE / f'date={TARGET_DATE}' / 'data.parquet'}")

Target date : 2026-04-27
Output path : ../datastore/openyld/TRACE/raw/date=2026-04-27/data.parquet


## 0.2 Authenticate

In [2]:
EMAIL = os.environ.get("OPENYLD_EMAIL")
PASSWORD = os.environ.get("OPENYLD_PASSWORD")

if not EMAIL or not PASSWORD:
    raise EnvironmentError("OPENYLD_EMAIL and OPENYLD_PASSWORD must be set in .env")

BASE_URL = "https://api.openyld.com"

auth_response = requests.post(
    f"{BASE_URL}/auth/login",
    json={"email": EMAIL, "password": PASSWORD}
)

if auth_response.status_code != 200:
    raise RuntimeError(f"Authentication failed ({auth_response.status_code}): {auth_response.text}")

token = auth_response.json().get("token")
headers = {"Authorization": f"Bearer {token}"}
print("Authentication successful.")

RuntimeError: Authentication failed (502): <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.24.0 (Ubuntu)</center>
</body>
</html>


# 1. Fetch All Records

Paginate through the endpoint until an empty page is returned.
Each page advances the sequence cursor to `last_provider_sequence + 1`.

In [ ]:
def fetch_page(date: str, sequence: int) -> list:
    url = f"{BASE_URL}/reported/historical/trace/{date}/{sequence}"
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        raise RuntimeError(f"Fetch failed ({response.status_code}): {response.text}")
    data = response.json()
    return data if isinstance(data, list) else data.get("data", [])


def fetch_all(date: str) -> pd.DataFrame:
    all_records = []
    sequence = 0
    page = 0

    while True:
        batch = fetch_page(date, sequence)
        if not batch:
            break
        all_records.extend(batch)
        page += 1
        print(f"  Page {page}: {len(batch)} records (running total: {len(all_records)})")

        last_seq = batch[-1].get("provider_sequence")
        if last_seq is None or last_seq <= sequence:
            break
        sequence = last_seq + 1

    return pd.DataFrame(all_records)


print(f"Fetching TRACE records for {TARGET_DATE}...")
df_raw = fetch_all(TARGET_DATE)
print(f"\nTotal records fetched: {len(df_raw)}")

# 2. Basic Cleaning

Fix known data quality issues before saving:
- `yield`: replace `-999999` sentinel with `NaN`
- `quantity`: cast from string to numeric
- `as_at`: parse to datetime

In [ ]:
if df_raw.empty:
    print(f"No records found for {TARGET_DATE}. Nothing to save.")
else:
    df = df_raw.copy()

    if "yield" in df.columns:
        df["yield"] = pd.to_numeric(df["yield"], errors="coerce")
        df.loc[df["yield"] == -999999, "yield"] = float("nan")

    if "quantity" in df.columns:
        df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

    if "as_at" in df.columns:
        df["as_at"] = pd.to_datetime(df["as_at"], errors="coerce")

    print(f"Records after cleaning: {len(df)}")
    print(f"Null counts:\n{df.isnull().sum()}")
    df.head()

# 3. Save to Parquet

Write to the partitioned datastore. If a file already exists for this
date it will be overwritten — safe to re-run if the API had issues mid-day.

In [ ]:
if not df_raw.empty:
    out_dir = DATASTORE / f"date={TARGET_DATE}"
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / "data.parquet"

    df.to_parquet(out_path, index=False)
    size_kb = out_path.stat().st_size / 1024
    print(f"Saved {len(df)} records to {out_path} ({size_kb:.1f} KB)")

# 4. Verify Saved File

In [ ]:
if not df_raw.empty:
    df_check = pd.read_parquet(out_path)
    print(f"Rows: {len(df_check)}, Columns: {list(df_check.columns)}")
    df_check.head()